# Joint comparison of candidate tilt mechanisms

This final notebook is run after the component notebooks pass QC. It asks which mechanisms retain within-eddy and between-eddy associations after adjustment. Direction and magnitude are analysed separately, AE and CE are kept separate, and whole eddies remain the clustering unit.

The goal is not to declare causality from a single coefficient. A mechanism is considered supported only when its directional, magnitude, timescale, regime and within-eddy predictions agree.


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd()
ANALYSIS_ROOT = HERE.parent
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))
if str(ANALYSIS_ROOT / "beta_effect_background_flow") not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT / "beta_effect_background_flow"))

import seacofs_tilt_tools as tilt
import mechanism_tools as mech

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df, _ = tilt.load_tilt_tables(paths)
df = mech.require_tilt_measurements(df)
print(f"Rows: {len(df):,}; measured tilts: {df.TiltDis.notna().sum():,}; eddies: {df.Eddy.nunique():,}")


Rows: 127,426; measured tilts: 105,621; eddies: 2,982


In [2]:
from beta_effect_background_flow.background_flow_tools import BackgroundConfig, load_background_cache

N2_CACHE = Path("/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset/tilt_mechanisms/n2_eddy_day_v3_core.parquet")
background = load_background_cache(BackgroundConfig()).drop(columns=["ic", "jc", "month"], errors="ignore")
n2 = pd.read_parquet(N2_CACHE)

data = mech.merge_one_to_one_or_many_to_one(df, background)
data = mech.merge_one_to_one_or_many_to_one(data, n2)
for depth in (200, 500):
    data[f"N2_{depth}m_s2"] = data[f"N2_{depth}m_core_s2"]
data = tilt.add_pv_gradient_terms(data, grid)
data = mech.add_stratification_proxies(data)
data = mech.add_background_shear(data)
data = mech.add_accumulated_shear(data, "ann_500", windows=(10, 20, 30))
data = tilt.add_region_labels(data, grid)
data = mech.add_topographic_regimes(data, shelf_depth=2000.0, dominance_ratio=2.0)
data["slope_mag"] = np.hypot(data.dhdx, data.dhdy)


In [3]:
# Predeclared mechanism variables. Add wind only after its cache passes QC.
mechanisms = [
    "beta", "N2_500m_s2", "N2_over_f2_500m", "Bu_proxy_500m",
    "ann_500_shear_mag_ms", "ann_500_accum_20d_mag_km",
    "slope_mag", "topo_plan_ratio_raw", "Rc", "h",
]
varying = ["ann_500_shear_mag_ms", "ann_500_accum_20d_mag_km", "Rc"]
data = mech.standardise_within_between(data, varying)
display(data[mechanisms].describe().T)


,count,mean,std,min,25%,50%,75%,max
beta,127424.0,-1.917380e-11,8.117010e-13,-2.082975e-11,-1.991064e-11,-1.908482e-11,-1.846449e-11,-1.755648e-11
N2_500m_s2,127426.0,7.593502e-05,1.585094e-05,4.543548e-05,6.410463e-05,7.402050e-05,8.636573e-05,3.229294e-04
N2_over_f2_500m,127426.0,1.208405e+04,4.452874e+03,5.331661e+03,8.674458e+03,1.117918e+04,1.468215e+04,6.546508e+04
Bu_proxy_500m,127426.0,8.376746e-02,9.260526e-02,3.315199e-03,2.963962e-02,5.526053e-02,1.027679e-01,3.514835e+00
ann_500_shear_mag_ms,127426.0,7.296612e-02,4.326004e-02,0.000000e+00,4.061567e-02,6.571631e-02,9.770739e-02,5.380698e-01
ann_500_accum_20d_mag_km,124444.0,5.363503e+01,3.675332e+01,8.788898e-02,2.541893e+01,4.751640e+01,7.439313e+01,4.291230e+02
slope_mag,127424.0,1.402150e-02,2.227736e-02,1.098551e-05,2.035181e-03,4.430007e-03,1.540749e-02,1.637708e-01
topo_plan_ratio_raw,127424.0,2.644932e+01,7.865258e+01,5.759771e-03,1.725192e+00,4.375227e+00,1.810903e+01,1.491268e+03
Rc,127426.0,7.862359e+01,3.444922e+01,1.484732e+01,5.290895e+01,7.246890e+01,9.816889e+01,2.845415e+02
h,127426.0,4.082367e+03,1.042393e+03,6.510651e+01,3.654601e+03,4.606874e+03,4.750918e+03,4.942000e+03


In [4]:
# Magnitude: compare standardized clustered models, then inspect effect sizes and CIs.
import statsmodels.formula.api as smf

model_sets = {
    "environment": ["beta", "N2_over_f2_500m", "Rc", "h"],
    "plus_shear": ["beta", "N2_over_f2_500m", "Rc", "h",
                   "ann_500_accum_20d_mag_km_between", "ann_500_accum_20d_mag_km_within"],
    "plus_topography": ["beta", "N2_over_f2_500m", "Rc", "h", "slope_mag", "topo_plan_ratio_raw",
                        "ann_500_accum_20d_mag_km_between", "ann_500_accum_20d_mag_km_within"],
}
fits = {}
for cyc, part in data.groupby("Cyc"):
    for name, columns in model_sets.items():
        use = part[["Eddy", "TiltDis", *columns]].replace([np.inf, -np.inf], np.nan).dropna().copy()
        for column in columns:
            sd = use[column].std()
            use[f"z_{column}"] = (use[column] - use[column].mean()) / sd if sd > 0 else 0
        formula = "np.log1p(TiltDis) ~ " + " + ".join(f"z_{c}" for c in columns)
        fits[(cyc, name)] = smf.gee(formula, groups="Eddy", data=use).fit()
        print(cyc, name, "rows", len(use), "eddies", use.Eddy.nunique())


AE environment rows 53607 eddies 1422
AE plus_shear rows 53607 eddies 1422
AE plus_topography rows 53607 eddies 1422
CE environment rows 52014 eddies 1531
CE plus_shear rows 52014 eddies 1531
CE plus_topography rows 52014 eddies 1531


In [5]:
for key, fit in fits.items():
    print("\n", key)
    display(pd.DataFrame({"estimate": fit.params, "ci_low": fit.conf_int()[0], "ci_high": fit.conf_int()[1]}))



 ('AE', 'environment')


,estimate,ci_low,ci_high
Intercept,2.986349,2.954053,3.018644
z_beta,-0.131188,-0.184450,-0.077926
z_N2_over_f2_500m,0.173810,0.119343,0.228276
z_Rc,-0.123819,-0.148467,-0.099171
z_h,-0.018305,-0.042817,0.006207



 ('AE', 'plus_shear')


,estimate,ci_low,ci_high
Intercept,2.986349,2.954713,3.017985
z_beta,-0.117637,-0.171201,-0.064072
z_N2_over_f2_500m,0.188068,0.133691,0.242445
z_Rc,-0.130735,-0.154832,-0.106637
z_h,-0.011311,-0.035980,0.013359
z_ann_500_accum_20d_mag_km_between,-0.065045,-0.092536,-0.037554
z_ann_500_accum_20d_mag_km_within,-0.018830,-0.035537,-0.002122



 ('AE', 'plus_topography')


...

,estimate,ci_low,ci_high
Intercept,2.812133,2.785141,2.839125
z_beta,-0.356709,-0.400790,-0.312627
z_N2_over_f2_500m,-0.087621,-0.129060,-0.046182
z_Rc,-0.064618,-0.087492,-0.041744
z_h,-0.029096,-0.054029,-0.004163



 ('CE', 'plus_shear')


,estimate,ci_low,ci_high
Intercept,2.812133,2.785952,2.838314
z_beta,-0.311433,-0.355993,-0.266873
z_N2_over_f2_500m,-0.035215,-0.077609,0.007180
z_Rc,-0.081842,-0.104436,-0.059248
z_h,-0.018570,-0.042925,0.005784
z_ann_500_accum_20d_mag_km_between,-0.113116,-0.137718,-0.088514
z_ann_500_accum_20d_mag_km_within,0.003732,-0.011660,0.019123



 ('CE', 'plus_topography')


,estimate,ci_low,ci_high
Intercept,2.812133,2.786247,2.838019
z_beta,-0.294295,-0.340155,-0.248435
z_N2_over_f2_500m,-0.017844,-0.061064,0.025376
z_Rc,-0.090300,-0.113039,-0.067561
z_h,-0.064522,-0.093979,-0.035066
z_slope_mag,-0.052151,-0.077775,-0.026527
z_topo_plan_ratio_raw,-0.037762,-0.069751,-0.005772
z_ann_500_accum_20d_mag_km_between,-0.104278,-0.128859,-0.079698
z_ann_500_accum_20d_mag_km_within,0.001612,-0.013850,0.017075


In [6]:
# Directional scorecard assembled from the component notebooks.
direction_metrics = [
    "ann_500_tilt_shear_offset",
    "ann_500_accum_20d_offset",
]
data["tilt_planetary_pv_offset"] = mech.signed_angle_difference(data.TiltDir, data.PV_grad_plan_theta)
data["tilt_topographic_pv_offset"] = mech.signed_angle_difference(data.TiltDir, data.PV_grad_topo_theta)
direction_metrics += ["tilt_planetary_pv_offset", "tilt_topographic_pv_offset"]
scorecard = mech.circular_offset_summary(data, direction_metrics, group=("Cyc", "ShelfRegime", "PVRegime"))
display(scorecard.sort_values(["Cyc", "ShelfRegime", "resultant_length"], ascending=[True, True, False]))


,Cyc,ShelfRegime,PVRegime,metric,eddies,mean_offset_deg,resultant_length,median_abs_offset_deg
10,AE,off_shelf,topographic,tilt_planetary_pv_offset,1224,-161.856367,0.329452,192.569651
2,AE,off_shelf,mixed,tilt_planetary_pv_offset,996,-172.497365,0.280772,184.720250
6,AE,off_shelf,planetary,tilt_planetary_pv_offset,524,-175.561068,0.238133,183.091162
1,AE,off_shelf,mixed,ann_500_accum_20d_offset,996,-105.398316,0.158796,207.180754
0,AE,off_shelf,mixed,ann_500_tilt_shear_offset,996,-121.279669,0.145468,207.396679
5,AE,off_shelf,planetary,ann_500_accum_20d_offset,524,-119.172209,0.133716,201.158059
9,AE,off_shelf,topographic,ann_500_accum_20d_offset,1224,-83.626268,0.129103,210.158194
4,AE,off_shelf,planetary,ann_500_tilt_shear_offset,524,-148.505811,0.102699,190.433139
8,AE,off_shelf,topographic,ann_500_tilt_shear_offset,1224,-90.836460,0.090642,204.383417
11,AE,off_shelf,topographic,tilt_topographic_pv_offset,1224,-62.110776,0.060111,193.112874


## Final evidence rubric

For each mechanism report:

1. **Direction:** Is the eddy-equal offset concentrated around the predicted direction?
2. **Magnitude:** Does forcing magnitude predict `TiltDis` with a scientifically meaningful effect?
3. **Timescale:** Does a plausible trailing window outperform instantaneous forcing?
4. **Within eddies:** Does the same eddy respond when forcing changes?
5. **Regime:** Does the relationship strengthen where the mechanism should dominate?
6. **Robustness:** Does it survive spatial blocks, tilt thresholds, depth choice and background definition?

Use language such as “supports,” “is consistent with,” or “does not support.” Reserve causal language for a coherent suite of predictions, not isolated significance.
